# Import Library

In [1]:
!pip install kaggle

In [2]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
%matplotlib inline

# Import Data

In [3]:
! mkdir ~/.kaggle
! cp kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json
! kaggle competitions download -c 114-ml-hw-2-store-sales-time-series-forecasting

100% 22.0M/22.0M [00:00<00:00, 107MB/s]



In [4]:
!unzip 114-ml-hw-2-store-sales-time-series-forecasting

Archive:  114-ml-hw-2-store-sales-time-series-forecasting.zip
  inflating: holidays_events.csv     
  inflating: oil.csv                 
  inflating: sample_submission.csv   
  inflating: stores.csv              
  inflating: test.csv                
  inflating: train.csv               
  inflating: transactions.csv        


# Time Series Model

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from google.colab import files

train = pd.read_csv("train.csv")
train = train[train['date'] >= '2017-01-01']
test = pd.read_csv("test.csv")
stores = pd.read_csv("stores.csv")
oil = pd.read_csv("oil.csv")
event = pd.read_csv("holidays_events.csv")
sub = pd.read_csv("sample_submission.csv")

oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill()
oil['oil_7d_ma'] = oil['dcoilwtico'].rolling(window=7, min_periods=1).mean().ffill().bfill()

event = event[event['transferred'] == False]
event = event.drop_duplicates(subset=['date'], keep='first')

train_df = train.merge(oil, on='date', how='left')
train_df = train_df.merge(stores, on='store_nbr', how='left')
train_df = train_df.merge(event, on='date', how='left')

test_df = test.merge(oil, on='date', how='left')
test_df = test_df.merge(stores, on='store_nbr', how='left')
test_df = test_df.merge(event, on='date', how='left')

for df in [train_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])
    df['Year'] = df['date'].dt.year
    df['Month'] = df['date'].dt.month
    df['Day'] = df['date'].dt.day
    df['Weekday'] = df['date'].dt.weekday
    df['DayOfYear'] = df['date'].dt.dayofyear
    df['is_payday'] = df['Day'].apply(lambda x: 1 if x == 15 or x >= 30 else 0)
    df['is_weekend'] = df['Weekday'].apply(lambda x: 1 if x >= 5 else 0)

    fill_cols = ['type_y', 'locale', 'locale_name', 'description', 'transferred', 'type_x']
    for c in fill_cols:
        if c in df.columns:
            df[c] = df[c].fillna('None')

cat_features = ['family', 'city', 'state', 'type_x', 'type_y', 'locale', 'locale_name', 'description', 'transferred']
for col in cat_features:
    le = LabelEncoder()
    combined = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
    le.fit(combined)
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))

drop_cols = ['id', 'date']
X_train = train_df.drop(columns=drop_cols + ['sales'], errors='ignore')
y_train = train_df['sales']
X_test = test_df.drop(columns=drop_cols + ['sales'], errors='ignore')

X_train = X_train.apply(pd.to_numeric, errors='coerce')
X_test = X_test.apply(pd.to_numeric, errors='coerce')
y_train_log = np.log1p(y_train)

model = XGBRegressor(
    n_estimators=1800,
    learning_rate=0.025,
    max_depth=9,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=7,
    random_state=42,
    tree_method='hist',
    n_jobs=-1
)
model.fit(X_train, y_train_log)

pdt_log = model.predict(X_test)
pdt = np.expm1(pdt_log)
pdt[pdt < 0] = 0

sub['sales'] = pdt
sub.to_csv('baseline.csv', index=False)
files.download('baseline.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>